In [0]:
USE e_commerce.silver;

In [0]:
%run "../src/utils/merge"

In [0]:
CREATE TABLE IF NOT EXISTS silver.t_order_x_item
USING DELTA
LOCATION 'abfss://silver@saexternaldatabricks.dfs.core.windows.net/t_order_x_item'
AS
SELECT
  o.order_id,
  oi.order_item_id,
  o.status,
  o.purchase_date,
  o.approved_date,
  o.carrier_date,
  o.delivered_customer_date,
  o.estimated_delivery_date,
  o.customer_id,
  oi.product_id,
  oi.seller_id,
  oi.shipping_limit_date,
  oi.price,
  oi.freight_value,
  current_timestamp() as ingestion_timestamp
FROM
  order o
INNER JOIN
  order_item oi
ON
  o.order_id = oi.order_id
LIMIT 0;

In [0]:
OPTIMIZE silver.t_order_x_item
ZORDER BY (order_id, order_item_id);

In [0]:
%python
df_order_x_item_updates = spark.sql("""
    SELECT
        o.order_id,
        oi.order_item_id,
        o.status,
        o.purchase_date,
        o.approved_date,
        o.carrier_date,
        o.delivered_customer_date,
        o.estimated_delivery_date,
        o.customer_id,
        oi.product_id,
        oi.seller_id,
        oi.shipping_limit_date,
        oi.price,
        oi.freight_value,
        current_timestamp() AS ingestion_timestamp
    FROM
        order o
    INNER JOIN
        order_item oi
    ON
        o.order_id = oi.order_id;
""")

In [0]:
%python
try:
    merge(
        df_source = df_order_x_item_updates,
        target_table = "e_commerce.silver.t_order_x_item",
        merge_key = ["order_id","order_item_id"],
        transforms = {

        },
        exclude_update_cols = []
    )
except Exception as e:
    print(e)

In [0]:
SELECT COUNT(1) FROM silver.t_order_x_item;